# 📝 텍스트 임베딩 과제 LV1(기초) — 임베딩·유사도·검색·군집 기초

> 이 단원에서 배운 **문장 임베딩**(문장→768차원 벡터)·**코사인 유사도**·**의미 검색**·**차원축소(UMAP)**·**군집화(KMeans)**·**실루엣 점수**·**벡터 연산(내적·노름·정규화)** 을 **한 문제에 하나씩** 확인하는 과제입니다.

## 풀이 방법
1. 맨 위 **제공 코드 셀**(라이브러리·임베딩 모델)을 먼저 실행하세요.
2. 각 문제의 **답안 셀**(`# 여기에 코드를 작성하세요`)에 코드를 채웁니다.
3. 바로 아래 **자가채점 셀**(`# [자가채점]`)을 실행해 `✅ 통과!` 가 뜨면 성공이에요.
4. 막히면 `힌트` 를 펼쳐 보세요.

- 데이터는 `data/news_headlines.csv`(뉴스 헤드라인 20건, 열: `headline` 헤드라인 문장, `category` 분야 — 정치·경제·스포츠·IT 각 5건) 를 씁니다.
- 임베딩은 이 단원 모델로 **직접 만듭니다**(20건이라 금방 끝나요). 문제 2에서 만든 임베딩 `embeddings` 를 뒤 문제들에서 그대로 씁니다.

화이팅!

아래 두 셀을 먼저 실행해 라이브러리와 임베딩 모델을 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리를 준비합니다.
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from umap import UMAP
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

이어서 이 단원에서 배운 **한국어 임베딩 모델**을 불러옵니다(문장→768차원 벡터). 처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.

In [ ]:
# [제공 코드] 이 단원에서 배운 한국어 임베딩 모델을 불러옵니다(문장->768차원 벡터).
model = SentenceTransformer('jhgan/ko-sroberta-multitask')

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `value_counts()` 로 분야 분포를 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·분야 분포
df = pd.read_csv('data/news_headlines.csv')
print("행·열 크기:", df.shape)
print("\n[앞 5행] head()"); display(df.head())
print("\n[열·자료형·결측] info()"); df.info()
print("\n[분야(category) 분포] value_counts()"); display(df["category"].value_counts().to_frame("건수"))

## 1. 헤드라인 한 개를 벡터로 — 임베딩 만들기
**배경**: 임베딩의 출발점은 **문장 하나를 벡터 하나로** 바꾸는 것입니다. 모델의 `encode` 에 문장을 넣으면 그 문장의 뜻을 담은 768차원 벡터가 나옵니다.

**요구사항**:
- `df` 의 **0번 헤드라인**(`df.loc[0, 'headline']`)을 `model.encode(...)` 로 임베딩해 변수 `vec` 에 담으세요.
- `vec.shape` 가 `(768,)` 인지 확인하세요(문장 한 개는 768차원 벡터 하나).

**예시**
```
vec.shape  →  (768,)
```
<details><summary>힌트</summary>

```text
접근방법:
- 헤드라인 문자열 하나를 임베딩 모델의 encode 에 넣어 벡터를 얻는다.

세부구현:
1. df 의 0번 행 headline 값을 꺼낸다
2. model 의 encode 에 그 문장을 넣어 vec 에 담는다
3. vec 의 shape 를 출력해 (768,) 인지 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert vec.shape == (768,)
print("✅ 문제1 통과!")

## 2. 헤드라인 20건을 한꺼번에 임베딩
**배경**: 분석하려면 헤드라인 **전부**를 벡터로 바꿔 둬야 합니다. `encode` 에 문장 **리스트**를 넣으면 `(문장 수, 768)` 모양의 행렬이 한 번에 나옵니다.

**요구사항**:
- `df['headline']` 의 헤드라인 20건을 리스트로 만들어 `model.encode(...)` 로 임베딩하고, 결과를 변수 `embeddings` 에 담으세요.
- `embeddings.shape` 가 `(20, 768)` 인지 확인하세요(헤드라인 20건 × 각 768차원).
- 이 `embeddings` 는 **뒤 문제(3~15번)에서 그대로** 사용합니다.

**예시**
```
embeddings.shape  →  (20, 768)
```
<details><summary>힌트</summary>

```text
접근방법:
- 헤드라인 열을 리스트로 바꿔 통째로 encode 에 넣으면 문장마다 한 행인 행렬이 나온다.

세부구현:
1. df 의 headline 열을 리스트로 만든다
2. model 의 encode 에 그 리스트를 넣어 embeddings 에 담는다
3. embeddings 의 shape 를 출력해 (20, 768) 인지 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert embeddings.shape == (20, 768)
print("✅ 문제2 통과!")

## 3. 코사인 유사도 — 같은 분야 두 헤드라인
**배경**: 두 문장이 얼마나 비슷한지는 **코사인 유사도**(두 벡터가 이루는 각도, 1에 가까울수록 비슷)로 잽니다. 먼저 **같은 분야**(둘 다 스포츠)인 10번·11번 헤드라인의 유사도를 재 봅니다.

**요구사항**:
- 문제 2의 `embeddings` 에서 10번·11번 벡터로 코사인 유사도를 구해 변수 `sim_same` 에 담으세요. `cosine_similarity(embeddings[10:11], embeddings[11:12])[0][0]` 로 값 하나를 꺼냅니다.
- `sim_same` 은 약 **0.31** 입니다. 모델 버전에 따라 소수 자리가 미세하게 달라질 수 있어 **`0.28 ~ 0.34` 범위**로 채점합니다(정확한 값이 아니라 범위·방향으로 봅니다).

**예시**
```
round(sim_same, 2)  →  0.31   (스포츠 ↔ 스포츠)
```
<details><summary>힌트</summary>

```text
접근방법:
- 두 헤드라인의 벡터를 코사인 유사도 함수에 넣어 값 하나를 꺼낸다.

세부구현:
1. embeddings 에서 10번·11번 벡터를 각각 한 행짜리로 고른다
2. cosine_similarity 에 두 벡터를 넣고 [0][0] 으로 값 하나를 꺼내 sim_same 에 담는다
3. 값을 출력해 약 0.31 인지 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert 0.28 < float(sim_same) < 0.34
print("✅ 문제3 통과!")

## 4. 코사인 유사도 — 다른 분야 두 헤드라인
**배경**: 이번엔 **다른 분야**(스포츠 vs IT)인 10번·15번 헤드라인의 유사도를 재고, 문제 3의 **같은 분야** 유사도와 비교합니다. 임베딩이 분야(뜻)의 차이를 담았다면 같은 분야가 더 높아야 합니다.

**요구사항**:
- 10번·15번 벡터로 코사인 유사도를 구해 변수 `sim_diff` 에 담으세요.
- `sim_diff` 는 약 **0.13** 입니다(**`0.08 ~ 0.18` 범위**로 채점).
- 문제 3의 `sim_same`(같은 분야)이 `sim_diff`(다른 분야)보다 **큰지** 확인하세요(같은 분야가 더 비슷) — 이 방향이 이 문제의 핵심입니다.

**예시**
```
round(sim_diff, 2)      →  0.13    (스포츠 ↔ IT)
sim_same > sim_diff     →  True    (같은 분야가 더 비슷)
```
<details><summary>힌트</summary>

```text
접근방법:
- 문제 3과 똑같은 방법으로 대상만 바꿔(10·15번) 유사도를 구하고, 같은 분야 값과 크기를 비교한다.

세부구현:
1. embeddings 에서 10번·15번 벡터를 한 행짜리로 고른다
2. cosine_similarity 로 값 하나를 꺼내 sim_diff 에 담는다
3. sim_same 과 sim_diff 를 출력해 같은 분야가 더 큰지 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert 0.08 < float(sim_diff) < 0.18
assert sim_same > sim_diff
print("✅ 문제4 통과!")

## 5. 의미 검색 — 질문에 가장 가까운 헤드라인 1건
**배경**: **의미 검색**은 질문을 임베딩해, 모든 문서 중 **뜻이 가장 가까운** 것을 찾는 것입니다. 질문의 단어가 헤드라인에 그대로 없어도 뜻이 맞으면 찾아냅니다.

**요구사항**:
- 질문 문자열 `query = '프로 스포츠 경기에서 우승'` 을 `model.encode([query])` 로 임베딩하세요.
- 질문 벡터와 `embeddings` 의 코사인 유사도 배열(길이 20)을 구해 변수 `scores` 에 담으세요(`cosine_similarity(query_vec, embeddings)[0]`).
- 유사도가 **가장 큰** 헤드라인의 인덱스를 정수로 변수 `top1` 에 담으세요. `top1` 은 **11**(스포츠 헤드라인)이 나와야 합니다.

**예시**
```
len(scores)  →  20
top1         →  11   (가장 가까운 헤드라인의 인덱스)
```
<details><summary>힌트</summary>

```text
접근방법:
- 질문을 문서와 같은 모델로 임베딩하고, 모든 문서와의 유사도를 구해 가장 큰 값의 위치를 찾는다.

세부구현:
1. query 를 리스트로 감싸 encode 해 질문 벡터를 만든다
2. cosine_similarity 로 질문 벡터와 embeddings 의 유사도 배열을 구해 scores 에 담는다
3. 유사도가 가장 큰 위치를 정수로 top1 에 담는다(가장 큰 값의 인덱스를 찾는 numpy 함수 사용)
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(scores) == 20
assert top1 == 11
print("✅ 문제5 통과!")

## 6. 의미 검색 — 질문에 가장 가까운 헤드라인 top3
**배경**: 이번엔 **다른 질문**으로 가장 가까운 헤드라인 **3건**을 순서대로 찾습니다. 상위 k개를 뽑는 것이 실제 검색과 더 가깝습니다.

**요구사항**:
- 질문 `query2 = '인공지능 기술과 반도체 신제품'` 을 임베딩하고, `embeddings` 와의 유사도 배열을 변수 `scores2` 에 담으세요.
- 유사도가 **큰 순서**로 정렬한 인덱스에서 **상위 3개**를 변수 `top3` 에 담으세요(`np.argsort(-scores2)[:3]`).
- `top3` 의 길이는 3이고, **첫 번째**(가장 가까운) 인덱스는 **15**(IT 헤드라인)가 나와야 합니다.

**예시**
```
len(top3)   →  3
top3[0]     →  15   (가장 가까운 헤드라인의 인덱스)
```
<details><summary>힌트</summary>

```text
접근방법:
- 질문을 임베딩해 유사도 배열을 구하고, 큰 값부터 정렬한 인덱스에서 앞 3개를 고른다.

세부구현:
1. query2 를 리스트로 감싸 encode 하고 유사도 배열 scores2 를 구한다
2. 유사도에 마이너스를 붙여 argsort 하면 큰 값이 앞으로 온다(내림차순 인덱스)
3. 그 인덱스의 앞 3개를 top3 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(top3) == 3
assert int(top3[0]) == 15
print("✅ 문제6 통과!")

## 7. 차원축소 — UMAP으로 768차원을 2차원으로
**배경**: 768차원 벡터는 눈으로 볼 수 없습니다. **UMAP** 은 각 점의 **가까운 이웃 관계가 유지되도록** 임베딩을 **2차원 좌표**로 줄여 줍니다(나중에 산점도로 그릴 수 있게).

**요구사항**:
- `UMAP(n_components=2, n_neighbors=5, min_dist=0.05, random_state=0)` 로 `embeddings` 를 2차원으로 변환해 결과를 변수 `coords` 에 담으세요(`random_state=0` 을 주면 매번 같은 좌표가 나옵니다).
- `coords.shape` 가 `(20, 2)` 인지 확인하세요(헤드라인 20건 × 각 2차원 좌표).

**예시**
```
coords.shape  →  (20, 2)
```
<details><summary>힌트</summary>

```text
접근방법:
- UMAP 을 2차원으로 설정하고 임베딩에 fit_transform 을 적용해 2D 좌표를 얻는다.

세부구현:
1. UMAP 을 n_components=2, n_neighbors=5, min_dist=0.05, random_state=0 으로 만든다
2. fit_transform 에 embeddings 를 넣어 coords 에 담는다
3. coords 의 shape 를 출력해 (20, 2) 인지 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert coords.shape == (20, 2)
print("✅ 문제7 통과!")

## 8. 한 헤드라인의 2차원 좌표 살펴보기
**배경**: UMAP 으로 줄인 `coords` 에서 **한 헤드라인의 좌표**는 한 행입니다. 0번 헤드라인의 2D 좌표를 꺼내, 좌표가 정말 **2개의 숫자**(x, y)로 되어 있는지 확인해 봅니다.

**요구사항**:
- 문제 7의 `coords` 에서 **0번 헤드라인의 좌표**를 꺼내 변수 `first_coord` 에 담으세요(`coords[0]`).
- `len(first_coord)` 가 **2** 인지 확인하세요(2차원이므로 좌표 값이 2개).

**예시**
```
len(first_coord)  →  2      (x, y 두 개의 좌표 값)
```
<details><summary>힌트</summary>

```text
접근방법:
- 2D 좌표 행렬에서 0번 행을 꺼내면 그 헤드라인의 (x, y) 좌표가 된다. 원소 개수를 센다.

세부구현:
1. coords 의 0번 행을 first_coord 에 담는다
2. len 으로 좌표 값의 개수를 확인한다(2여야 함)
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(first_coord) == 2
print("✅ 문제8 통과!")

## 9. 군집화 — KMeans로 헤드라인 그룹 만들기
**배경**: 정답(분야) 없이 비슷한 헤드라인끼리 묶는 것이 **군집화**입니다. **KMeans** 는 미리 정한 개수(k)의 중심을 찾아 각 헤드라인을 가장 가까운 중심에 배정합니다. 768차원 원본은 노이즈가 커서 군집이 흐릿하므로, 문제 7에서 만든 **2차원 좌표 `coords`** 위에서 묶습니다. 분야는 4개지만 **몇 개로 나누는 게 좋은지는 실루엣으로 정합니다**(문제 12) — 여기서는 우선 **k=3** 으로 나눠 봅니다.

**요구사항**:
- 문제 7의 `coords` 를 사용합니다. `KMeans(n_clusters=3, random_state=0, n_init=10)` 로 모델을 만들고 `fit_predict(coords)` 로 각 헤드라인의 군집 번호를 구해 변수 `labels` 에 담으세요.
- `labels` 의 길이는 **20**(헤드라인 수), 서로 다른 군집 번호의 개수(`len(set(labels))`)는 **3** 이어야 합니다.

**예시**
```
len(labels)        →  20
len(set(labels))   →  3    (군집이 3개)
```
<details><summary>힌트</summary>

```text
접근방법:
- 2차원 좌표 coords 에 KMeans 를 학습시켜 각 헤드라인의 군집 번호를 얻는다. 재현을 위해 random_state 를 고정한다.

세부구현:
1. KMeans 를 n_clusters=3, random_state=0, n_init=10 으로 만든다
2. fit_predict 에 coords 를 넣어 labels 에 담는다
3. labels 의 길이와 서로 다른 군집 번호의 개수를 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(labels) == 20
assert len(set(labels)) == 3
print("✅ 문제9 통과!")

## 10. 같은 분야 헤드라인이 같은 군집에 묶였나
**배경**: 군집이 분야를 잘 잡았다면 **같은 분야** 헤드라인은 **같은 군집**에 묶여야 합니다. 문제 9의 `labels` 로, 스포츠 헤드라인인 10번과 11번이 같은 군집인지 확인해 봅니다.

**요구사항**:
- 문제 9의 `labels` 를 그대로 사용하세요.
- 10번 헤드라인의 군집 번호(`labels[10]`)와 11번 헤드라인의 군집 번호(`labels[11]`)가 **같은지** 비교한 결과(참/거짓)를 변수 `same_cluster` 에 담으세요.
- 두 헤드라인은 모두 스포츠라 같은 군집에 묶여, `same_cluster` 는 **참(True)** 이어야 합니다.

**예시**
```
same_cluster  →  True   (10번·11번이 같은 군집)
```
<details><summary>힌트</summary>

```text
접근방법:
- 두 헤드라인의 군집 번호를 꺼내 서로 같은지 비교한다.

세부구현:
1. labels 에서 10번·11번의 군집 번호를 각각 꺼낸다
2. 두 값이 같은지 비교(==)한 결과를 same_cluster 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert bool(same_cluster) == True
assert same_cluster == (labels[10] == labels[11])   # 값을 지어내지 않고 labels 로 구했는지
print("✅ 문제10 통과!")

## 11. 군집이 잘 나뉘었나 — 실루엣 점수
**배경**: 군집이 **얼마나 잘 나뉘었는지**는 **실루엣 점수**로 잽니다(−1~1, 클수록 좋음). 같은 군집끼리 가깝고 다른 군집과 멀수록 높습니다. 문제 9의 k=3 군집이 얼마나 잘 나뉘었는지 확인해 봅니다.

**요구사항**:
- 문제 7의 `coords` 와 문제 9의 `labels` 로 실루엣 점수를 구해 변수 `sil` 에 담으세요(`silhouette_score(coords, labels)`).
- `sil` 은 대략 **0.5 안팎**의 양수 값이 나옵니다(정확한 값이 아니라 범위로 채점합니다).

**예시**
```
0.3 < sil < 0.8   →  True   (양수이고 0.5 안팎)
```
<details><summary>힌트</summary>

```text
접근방법:
- 2차원 좌표와 군집 라벨을 실루엣 점수 함수에 넣는다. 값이 클수록 군집이 뚜렷하게 나뉜 것.

세부구현:
1. silhouette_score 에 coords 와 labels 를 넣어 sil 에 담는다
2. sil 을 출력해 양수인지 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert 0.3 < sil < 0.8
print("✅ 문제11 통과!")

## 12. 몇 개로 나눌까 — k=3 과 k=6 실루엣 비교
**배경**: 군집을 **몇 개(k)로 나눌지**는 실루엣 점수로 비교해 정합니다. 문제 7의 `coords` 위에서 k=3 과 k=6 으로 각각 군집화해 실루엣 점수를 재고, **너무 잘게 쪼개면 오히려 나빠지는지** 확인해 봅니다.

**요구사항**:
- 문제 7의 `coords` 를 사용합니다. `KMeans(n_clusters=3, random_state=0, n_init=10)` 로 군집화한 라벨의 실루엣 점수를 변수 `sil3` 에 담으세요(문제 11의 값과 같습니다).
- `KMeans(n_clusters=6, random_state=0, n_init=10)` 로 군집화한 라벨의 실루엣 점수를 변수 `sil6` 에 담으세요.
- 두 값을 비교해, k=3 쪽이 더 큰지(`sil3 > sil6`)를 변수 `k3_better` 에 담으세요. 이 데이터에서는 **k=3 이 더 큽니다**(`k3_better` 는 참) — 헤드라인이 20건뿐인데 6개로 쪼개면 군집이 잘게 부서집니다.

**예시**
```
k3_better  →  True   (k=3 의 실루엣이 k=6 보다 큼)
```
<details><summary>힌트</summary>

```text
접근방법:
- 같은 coords 에 k=3, k=6 으로 각각 군집화해 실루엣 점수를 구하고, 두 점수의 크기를 비교한다.

세부구현:
1. k=3 으로 fit_predict 한 라벨로 silhouette_score 를 구해 sil3 에 담는다
2. k=6 으로 fit_predict 한 라벨로 silhouette_score 를 구해 sil6 에 담는다
3. sil3 가 sil6 보다 큰지 비교한 결과를 k3_better 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert bool(k3_better) == True
assert k3_better == (sil3 > sil6)   # sil3·sil6 를 실제로 구해 비교했는지
assert 0.3 < sil3 < 0.8 and 0.2 < sil6 < 0.8
print("✅ 문제12 통과!")

---
# 벡터 연산 — 수식으로 직접 계산해 보기

여기까지는 `cosine_similarity` 가 알아서 계산해 주었습니다. 마지막 세 문제에서는 **그 함수가 안에서 무슨 계산을 하는지** 직접 해 봅니다. 교안 3-1 절의 식 그대로입니다.

## 13. 내적 · 길이 · 코사인 유사도 손으로 계산하기
**배경**: 코사인 유사도는 **내적 ÷ (길이 × 길이)** 입니다. 768차원은 눈으로 못 따라가니 **2차원 벡터 두 개**로 식을 그대로 옮겨 보고, 라이브러리 값과 같은지 맞춰 봅니다.

**요구사항**:
- `a_vec = [3, 4]`, `b_vec = [4, 3]` 두 리스트를 만드세요.
- 두 벡터의 **내적**(같은 칸끼리 곱해 전부 더한 값)을 변수 `dot_ab` 에 담으세요.
- 각 벡터의 **길이(노름)** — 각 칸을 제곱해 더한 뒤 제곱근 — 을 변수 `norm_a`, `norm_b` 에 담으세요.
- 위 셋으로 **코사인 유사도**를 계산해 변수 `cos_ab` 에 담으세요.
- 같은 계산을 **함수 `my_cos(u, v)`** 로도 만드세요 — 두 리스트를 받아 코사인 유사도를 돌려주며, **칸이 몇 개든**(2개든 3개든) 동작해야 합니다.
- `cosine_similarity` 없이 **파이썬 기본 연산**(`sum`, `**`)만으로 구합니다.

**예시**
```
dot_ab            →  두 벡터의 내적 (숫자 하나)
norm_a            →  a_vec 의 길이 (숫자 하나)
cos_ab            →  dot_ab / (norm_a * norm_b)
my_cos([1, 0], [0, 1])  →  0.0   (직각인 두 벡터)
```
<details><summary>힌트</summary>

```text
접근방법:
- 내적·길이·코사인을 순서대로 구한다. 제곱근은 ** 0.5 로 낼 수 있다.
- 함수는 그 세 줄을 그대로 옮기되, 길이를 len(u) 로 받아 어떤 차원에서도 돌게 만든다.

세부구현:
1. 두 리스트를 만든다
2. 같은 위치끼리 곱한 값을 모두 더해 내적을 구한다(range 와 sum 을 함께 쓰면 짧다)
3. 각 칸을 제곱해 더한 뒤 0.5 제곱해 길이를 구한다
4. 내적을 두 길이의 곱으로 나눈다
5. 같은 계산을 my_cos(u, v) 함수 안에 넣는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(cos_ab - float(cosine_similarity([a_vec], [b_vec])[0][0])) < 1e-9
assert abs(dot_ab - norm_a * norm_b * cos_ab) < 1e-9   # 세 값의 아귀가 맞는지
assert abs(norm_a - norm_b) < 1e-9                     # 이 두 벡터는 길이가 같다
assert 20 < dot_ab < 30 and 4 < norm_a < 6
assert abs(my_cos(a_vec, b_vec) - cos_ab) < 1e-9
assert abs(my_cos([1, 0], [0, 1]) - 0.0) < 1e-9        # 직각이면 0
assert abs(my_cos([1, 2, 2], [2, 4, 4]) - 1.0) < 1e-9  # 3칸이어도, 방향이 같으면 1
print("✅ 문제13 통과!")

## 14. 768차원 임베딩으로 같은 계산 하기
**배경**: 식에는 차원 수의 제한이 없습니다. 문제 13과 **똑같은 계산**을 768칸짜리 실제 임베딩에 하면 문제 3에서 `cosine_similarity` 로 구한 값이 그대로 나와야 합니다.

**요구사항**:
- 문제 2의 `embeddings` 에서 10번·11번 벡터를 꺼내 씁니다.
- 두 벡터의 내적을 **`@` 연산자**로 구해 변수 `dot_10_11` 에 담으세요(`embeddings[10] @ embeddings[11]`).
- 10번 벡터의 길이를 `np.linalg.norm` 으로 구해 변수 `norm_10` 에, 11번 것을 `norm_11` 에 담으세요.
- 세 값으로 코사인 유사도를 계산해 변수 `cos_manual` 에 담으세요.
- 문제 3의 `sim_same` 과 **같은 값**이 나오는지 확인하세요.

**예시**
```
cos_manual  →  dot_10_11 / (norm_10 * norm_11)   (sim_same 과 같은 값)
```
<details><summary>힌트</summary>

```text
접근방법:
- 문제 13과 같은 식을 numpy 함수로 옮긴다. 길이는 1 이 아니다(이 모델은 정규화하지 않는다).

세부구현:
1. embeddings 에서 10번·11번 벡터를 꺼낸다
2. @ 로 내적을, np.linalg.norm 으로 각 길이를 구한다
3. 내적을 두 길이의 곱으로 나눠 cos_manual 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(cos_manual - float(sim_same)) < 1e-5      # 라이브러리 값과 일치해야 한다
assert abs(dot_10_11 - float(embeddings[10] @ embeddings[11])) < 1e-4
assert norm_10 > 1.5 and norm_11 > 1.5               # 이 모델은 길이 1 로 정규화하지 않는다
print("✅ 문제14 통과!")

## 15. L2 정규화 — 길이를 1 로 맞추면 내적이 곧 코사인
**배경**: 벡터를 **자기 길이로 나누면** 방향은 그대로인 채 길이가 1 이 됩니다(L2 정규화). 그러면 코사인 식의 분모가 1 이 되어 **내적만 계산해도 코사인 값이 나옵니다** — 벡터 DB 가 임베딩을 미리 정규화해 두는 이유입니다.

**요구사항**:
- 문제 14의 `norm_10`·`norm_11` 을 이용해 10번·11번 벡터를 각각 길이 1 로 만들어 변수 `unit_10`, `unit_11` 에 담으세요(벡터를 자기 길이로 나눕니다).
- 두 단위 벡터의 **내적**을 `@` 로 구해 변수 `dot_unit` 에 담으세요.
- `dot_unit` 이 문제 14의 `cos_manual` 과 같은지 확인하세요.

**예시**
```
np.linalg.norm(unit_10)  →  1.0
dot_unit                 →  cos_manual 과 같은 값
```
<details><summary>힌트</summary>

```text
접근방법:
- 정규화는 나눗셈 한 번이다. numpy 배열은 숫자로 나누면 모든 칸이 한꺼번에 나뉜다.

세부구현:
1. embeddings[10] 을 norm_10 으로 나눠 unit_10 에 담는다(11번도 같은 방식)
2. @ 로 두 단위 벡터의 내적을 구해 dot_unit 에 담는다
3. cos_manual 과 값을 비교해 본다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(float(np.linalg.norm(unit_10)) - 1.0) < 1e-5
assert abs(float(np.linalg.norm(unit_11)) - 1.0) < 1e-5
assert abs(dot_unit - float(cos_manual)) < 1e-5     # 정규화하면 내적 == 코사인
print("✅ 문제15 통과!")